# House Graph Generator Using GraphRAG and Neo4j
## Part 1: Creating the Graph Corpus using Neo4j

In [ ]:
# This cell is not needed if you have pip installed topologicpy
import sys
sys.path.append("C:/Users/sarwj/OneDrive - Cardiff University/Documents/GitHub/topologicpy/src")

## 1. Import the needed libraries

In [ ]:
from topologicpy.Neo4j import Neo4j
from topologicpy.Vertex import Vertex
from topologicpy.Topology import Topology
from topologicpy.Dictionary import Dictionary
from topologicpy.Graph import Graph
from topologicpy.Helper import Helper
from topologicpy.GraphDB import GraphDB
from topologicpy.Neo4j import Neo4j

## 2. Check the TopologicPy version

In [ ]:
print("This tutorial requires topologicpy version 0.9.31 or newer.")
print(Helper.Version())

## 3. Set your renderer:
* Visual studio code: "vscode"
* Google Colab: "colab"
* Browser: "browser"

In [ ]:
renderer = "vscode"

## 4. Utility function to find files at a path

In [ ]:
from pathlib import Path

def file_paths_in_folder(folder_path, recursive=False):
    folder = Path(folder_path)

    if not folder.is_dir():
        raise ValueError(f"Not a valid folder: {folder_path}")

    pattern = "**/*" if recursive else "*"

    return [
        str(path)
        for path in folder.glob(pattern)
        if path.is_file()
    ]


## 5. Set the path to the JSON graphs folder

In [ ]:
folder_path = r"C:\Users\sarwj\OneDrive - Cardiff University\IAAC\2025-26\S3 - Buildings As Graphs\notebooks\Supporting Files\dataset_GraphRAG"

## 6. Import the JSON graphs to TopologicPy

In [ ]:
# There are 7 room types, stored numerically as 0-6. These correspond to living room (0), bedroom (1), bathroom (2), kitchen (3), window (4), door (5), and front door (6), respectively.

room_mapping = { 0: "living room",
                 1: "bedroom",
                 2: "bathroom",
                 3: "kitchen",
                 4: "window",
                 5: "door",
                 6: "front door"
               }
graphs = []
file_paths = file_paths_in_folder(folder_path, recursive=False)
for i, file_path in enumerate(file_paths):
    g = Graph.ByJSONPath(file_path)
    graph_dict = Topology.Dictionary(g)
    graph_id = "graph_"+(str(i).zfill(3))
    bedrooms = Dictionary.ValueAtKey(graph_dict, "bedrooms")
    graph_dict = Dictionary.SetValuesAtKeys(graph_dict, ["graph_id", "label"], [graph_id, bedrooms])
    g = Topology.SetDictionary(g, graph_dict)
    verts = Graph.Vertices(g)
    for j, v in enumerate(verts):
        vertex_id = graph_id+"_vertex_"+(str(j).zfill(4))
        d = Topology.Dictionary(v)
        d = Dictionary.SetValuesAtKeys(d, ["vertex_id", "vertex_size", "label"], [vertex_id, 15, room_mapping.get(Dictionary.ValueAtKey(d, "type"), "Unknown")])
        v = Topology.SetDictionary(v, d)
    
    graphs.append(g)


## 7. Do a visual check

In [ ]:
for i in range(38,40,1):
    graph = graphs[i]
    print(graph)
    print(file_paths[i])
    Topology.Show(graph,
                vertexSizeKey="vertex_size",
                showVertexLabel=True,
                vertexLabelKey="label",
                backgroundColor="white",
                camera = [0,0,5],
                renderer=renderer)

In [ ]:
verts = Graph.Vertices(graphs[-1])
for v in verts:
    d = Topology.Dictionary(v)
    print(Dictionary.Keys(d), Dictionary.Values(d))

## 8. Connect to your Neo4j instance. Make sure it is running first:
* If using the Desktop version, open Neo4j Desktop
* If using the online version (Aura), go to your console at: https://console.neo4j.io/

In [ ]:
from getpass import getpass

url = input("URL: ")
username = input("Username: ")
password = getpass("Password: ")

gdb_manager = Neo4j.Manager(url=url, username=username, password=password, database=None, silent=False)
print(gdb_manager)

In [ ]:
# -----------------------------------------------------------------------------
# 1. Create / connect to Graph DB
# -----------------------------------------------------------------------------

graphdb = GraphDB.ByParameters(
    provider="neo4j",
    manager=gdb_manager
)

# Clear test DB
ok = GraphDB.EmptyDatabase(graphdb)
print("EmptyDatabase:", ok)
# -----------------------------------------------------------------------------
# 2. Ensure schema
# -----------------------------------------------------------------------------

ok = GraphDB.EnsureSchema(graphdb)
print("EnsureSchema:", ok)

In [ ]:
# -----------------------------------------------------------------------------
# 4. Upsert graphs into GraphDB
# -----------------------------------------------------------------------------

total = len(graphs)
for i, g in enumerate(graphs):
    graph_id = GraphDB.UpsertGraph(graphdb = graphdb,
                                   graph = g,
                                   graphIDKey = "graph_id",
                                   vertexIDKey = "id",
                                   vertexLabelKey = "label",
                                   defaultVertexLabel = "Node",
                                   vertexCategoryKey = "category",
                                   defaultVertexCategory = "Node",
                                   edgeLabelKey = "label",
                                   defaultEdgeLabel = "CONNECTED_TO",
                                   edgeCategoryKey = "category",
                                   defaultEdgeCategory = "Edge",
                                   bidirectional = True,
                                   overwrite = False,
                                   mantissa = 6,
                                   silent = False
                                   )
    print(f"{i+1}/{total}: {graph_id}")

print("Done Upserting Graphs.")

In [ ]:
# -----------------------------------------------------------------------------
# 5. Retrieve graph by ID
# -----------------------------------------------------------------------------

g2 = GraphDB.GraphByID(graphdb, graph_id)

print("Graph retrieved:", g2 is not None)

# -----------------------------------------------------------------------------
# 6. Corpus analytics tests
# -----------------------------------------------------------------------------

pairs = GraphDB.FetchAllPairs(graphdb)
print("Pairs:")
for row in pairs[:10]:
    print(row)

candidate_counts = GraphDB.CandidateCountsForLabels(
    graphdb,
    ["living room"]
)

print("\nCandidate Counts:")
for row in candidate_counts[:10]:
    print(row)

max_neighbors = GraphDB.MaxNeighborsForLabel(
    graphdb,
    "door"
)

print("\nMax Neighbors for door:")
print(max_neighbors)

max_neighbors = GraphDB.MaxNeighborsForLabel(
    graphdb,
    "kitchen"
)

print("\nMax Neighbors for kitchen:")
print(max_neighbors)

max_neighbors = GraphDB.MaxNeighborsForLabel(
    graphdb,
    "bathroom"
)

print("\nMax Neighbors for bathroom:")
print(max_neighbors)



example = GraphDB.FindBestExampleForLabel(
    graphdb,
    "kitchen"
)

print("\nBest Example for Kitchen:")
print(example)

# -----------------------------------------------------------------------------
# 7. List stored graphs
# -----------------------------------------------------------------------------

graphs = GraphDB.ListGraphs(graphdb)

print("\nStored Graphs:")
for g in graphs:
    print(g)